# Milestone 3: pure-greedy re-evaluation, class-balanced v2, draft model, KV cache, tiled attention, speculative decoding
Kaggle: **GPU T4 x2, Internet ON**, Run All. About 60-75 min.

**Before running:** put your HuggingFace adapter id in the first code cell.

In [ ]:
HF_ADAPTER_V1 = 'YOUR_HF_USERNAME/trialmatch-qwen25-1p5b-lora'   # <-- change this

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
assert 'YOUR_HF_USERNAME' not in HF_ADAPTER_V1, 'set HF_ADAPTER_V1 first'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf trial-eligibility-transformer
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd /kaggle/working/trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q tests   # includes KV / speculative / tiled-attention tests on tiny models
!python scripts/prepare_data.py | tail -3

## 1. Re-evaluate with pure greedy decoding
Qwen's default generation config adds a repetition penalty even when sampling is off. Our KV-cache and speculative decoders are pure greedy, so all evaluations are redone with `repetition_penalty=1.0` (a decoding-consistency fix, not tuning).

In [ ]:
!python scripts/evaluate.py --split val --tag baseline_val_greedy --model Qwen/Qwen2.5-1.5B-Instruct
!python scripts/evaluate.py --split val --tag ft_v1_val_greedy --adapter {HF_ADAPTER_V1}
!python scripts/evaluate.py --split test --final-test --tag baseline_test_greedy --model Qwen/Qwen2.5-1.5B-Instruct
!python scripts/evaluate.py --split test --final-test --tag ft_v1_test_greedy --adapter {HF_ADAPTER_V1}

## 2. v2: class-balanced fine-tuning (train split oversampled, 2 epochs), selected on validation

In [ ]:
!python scripts/train_lora.py --balance sqrt --epochs 2 --output-dir outputs/lora_qwen25_1p5b_v2

In [ ]:
import glob, subprocess, json
ck = sorted(glob.glob('outputs/lora_qwen25_1p5b_v2/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
assert ck, 'v2 training failed'
tags = []
for i, c in enumerate(ck, 1):
    tags.append(f'ft_v2_ep{i}_val')
    subprocess.run(['python', 'scripts/evaluate.py', '--adapter', c, '--split', 'val', '--tag', tags[-1]], check=True)
subprocess.run(['python', 'scripts/select_checkpoint.py', '--run', 'outputs/lora_qwen25_1p5b_v2', '--tags', *tags,
                '--selection-file', 'checkpoint_selection_v2.json'], check=True)

In [ ]:
# Final model choice: higher VALIDATION macro-F1 (rule fixed in advance)
v1 = json.load(open('results/ft_v1_val_greedy/metrics.json'))['overall']['classification']
sel = json.load(open('results/checkpoint_selection_v2.json'))['selected']
FINAL = 'v2' if sel['val_macro_f1'] > v1['macro_f1'] else 'v1'
FINAL_ADAPTER = 'outputs/lora_qwen25_1p5b_v2_best' if FINAL == 'v2' else HF_ADAPTER_V1
BALANCE = 'sqrt' if FINAL == 'v2' else 'none'
json.dump({'rule': 'higher validation macro-F1 under pure greedy decoding',
           'v1': {'adapter': HF_ADAPTER_V1, 'val_macro_f1': v1['macro_f1'], 'val_accuracy': v1['accuracy']},
           'v2': {'adapter': 'outputs/lora_qwen25_1p5b_v2_best', 'val_macro_f1': sel['val_macro_f1'], 'val_accuracy': sel['val_accuracy'], 'epoch': sel['epoch']},
           'final': FINAL}, open('results/final_model.json', 'w'), indent=2)
print(open('results/final_model.json').read())
if FINAL == 'v2':
    subprocess.run(['python', 'scripts/evaluate.py', '--split', 'test', '--final-test', '--tag', 'ft_v2_test_greedy', '--adapter', FINAL_ADAPTER], check=True)
FINAL_TEST_TAG = 'ft_v2_test_greedy' if FINAL == 'v2' else 'ft_v1_test_greedy'

In [ ]:
subprocess.run(['python', 'scripts/error_analysis.py', f'results/{FINAL_TEST_TAG}'], check=True)
tags, names = ['baseline_test_greedy', 'ft_v1_test_greedy'], ['Zero-shot', 'LoRA v1']
if FINAL == 'v2':
    tags.append('ft_v2_test_greedy'); names.append('LoRA v2 (balanced)')
subprocess.run(['python', 'scripts/make_figures.py', '--cm', 'baseline_test_greedy', *tags[1:],
                '--compare', *tags, '--compare-names', *names,
                '--train-log', 'outputs/lora_qwen25_1p5b_v2/train_log.json'], check=True)

## 3. Draft model for speculative decoding: Qwen2.5-0.5B-Instruct (same tokenizer), LoRA-tuned on the same train split (1 epoch)

In [ ]:
!python scripts/train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --epochs 1 --balance {BALANCE} --output-dir outputs/lora_draft_0p5b
DRAFT_ADAPTER = sorted(glob.glob('outputs/lora_draft_0p5b/checkpoint-*'))[-1]
!python scripts/evaluate.py --split val --tag draft_0p5b_val --model Qwen/Qwen2.5-0.5B-Instruct --adapter {DRAFT_ADAPTER}
!python scripts/evaluate.py --split val --tag draft_0p5b_base_val --model Qwen/Qwen2.5-0.5B-Instruct

## 4. Task 4: attention (reference vs tiled online-softmax vs fused SDPA; plus the whole model running on tiled attention)

In [ ]:
!python scripts/bench_attention.py --adapter {FINAL_ADAPTER}

## 5. Task 3: KV-cached vs no-cache greedy decoding (fp16 benchmark grid + fp32 exactness)

In [ ]:
!python scripts/bench_kv.py --adapter {FINAL_ADAPTER} --dtype float16
!python scripts/bench_kv.py --adapter {FINAL_ADAPTER} --dtype float32 --no-grid

## 6. Task 5: speculative decoding (base vs tuned 0.5B draft, k = 2/4/6/8) + fp32 exactness

In [ ]:
!python scripts/bench_spec.py --adapter {FINAL_ADAPTER} --drafts base=Qwen/Qwen2.5-0.5B-Instruct tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 2 4 6 8 --dtype float16
!python scripts/bench_spec.py --adapter {FINAL_ADAPTER} --drafts tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 4 --dtype float32 --n-task 10 --n-summary 3

In [ ]:
!python scripts/make_bench_figures.py
for t in ['baseline_test_greedy', 'ft_v1_test_greedy'] + (['ft_v2_test_greedy'] if FINAL == 'v2' else []) + ['ft_v1_val_greedy', 'draft_0p5b_val', 'draft_0p5b_base_val']:
    o = json.load(open(f'results/{t}/metrics.json'))['overall']
    print(f"{t:24s} acc={o['classification']['accuracy']:.3f} mF1={o['classification']['macro_f1']:.3f} evF1={o['evidence']['micro']['f1']:.3f}")

## 7. Download `m3_results.zip` (results + figures + benchmark logs) and `m3_adapters.zip` (v2 + draft adapters)

In [ ]:
!for d in outputs/*/; do [ -f $d/train_log.json ] && cp $d/train_log.json results/train_log_$(basename $d).json && cp $d/train_config.json results/train_config_$(basename $d).json; done
!zip -qr /kaggle/working/m3_results.zip results data/splits
!cd outputs && zip -qr /kaggle/working/m3_adapters.zip lora_qwen25_1p5b_v2_best $(basename $(dirname {DRAFT_ADAPTER}))/$(basename {DRAFT_ADAPTER})
!ls -la /kaggle/working